In [4]:
#Data handling
import numpy as np
import pandas as pd
import ast # standard library, no install needed -  for parsing springified lists

# Text vectorization and similarity
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

# Text preprocessing (even thougj installing this package is not really needed but it is neceessary
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

#Visualization package
import matplotlib as pyplot
import seaborn as sns


In [5]:
#nltk.download('stopwords')
#nltk.download('wordnet')
#nltk.download('omw-1.4')

In [6]:
#I think one of the best way to install a package is this method. It guarantees installation directly to the exact environment of the notebook.
#import sys
#!{sys.executable} -m pip install seaborn


In [7]:
#loead dataset in the environment
df = pd.read_csv('/Users/user/Downloads/Auspify Technology/Dataset.csv')

In [8]:
print(df.shape)

(8790, 10)


In [39]:
print(df.columns.tolist())

['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in']


In [40]:
print(df.isnull().sum()) #check for missing value in key columns

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64


**Building the Metadata Soup**

In [11]:
#Even no NAN value was found, we can still defensively fill up missing values using the codee below
df['director'] = df['director'].fillna('')
df['listed_in'] = df['listed_in'].fillna('')

In [12]:
#Build a combined text feature per title
df['content_features'] = df['listed_in'] + ' ' + df['director'] + ' ' + df['type']

In [13]:
#lets check our soup
df[['title', 'content_features']].head()

,title,content_features
0,Dick Johnson Is Dead,Documentaries Kirsten Johnson Movie
1,Ganglands,"Crime TV Shows, International TV Shows, TV Act..."
2,Midnight Mass,"TV Dramas, TV Horror, TV Mysteries Mike Flanag..."
3,Confessions of an Invisible Girl,"Children & Family Movies, Comedies Bruno Garot..."
4,Sankofa,"Dramas, Independent Movies, International Movi..."


**Convert text to Vector**

In [14]:
from sklearn.feature_extraction.text import CountVectorizer
#CountVectorizer works well here since content_features is a controlled vocabulary
count_vec = CountVectorizer(stop_words = 'english')
count_matrix = count_vec.fit_transform(df['content_features'])
print(count_matrix.shape)

(8790, 6472)


**calculating the content similarity score**

In [15]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(count_matrix, count_matrix)
print(cosine_sim.shape)

(8790, 8790)


**Generating Recommendation for selected title**

In [16]:
# I start by building a reverse lookup: title -> index
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim,df=df, indices=indices, top_n=10):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    title_indices = [i[0] for i in sim_scores]
    return df[['title', 'listed_in', 'director']].iloc[title_indices]

**Testing the model**

In [17]:
get_recommendations('Dick Johnson Is Dead')

,title,listed_in,director
110,9to5: The Story of a Movement,Documentaries,Not Given
151,Headspace: Unwind Your Mind,Documentaries,Not Given
235,Ya no estoy aquí: Una conversación entre Guill...,Documentaries,Not Given
422,Bill Hicks: Reflections,Documentaries,Not Given
3293,Monty Python Conquers America,Documentaries,Will Yapp
4809,Down The Fence,Documentaries,M.J. Isakson
5072,Holy Hell,Documentaries,Will Allen
5665,Perfect Bid: The Contestant Who Knew Too Much,Documentaries,C.J. Wallis
6466,Creating an Army of the Dead,Documentaries,Not Given
6789,Free to Play,Documentaries,Not Given


In [19]:
def genre_overlap_score(title, df=df, indices=indices, top_n=10):
    idx = indices[title]
    input_genres = set(df.loc[idx, 'listed_in'].split(', '))
    recs = get_recommendations(title, top_n=top_n)

    overlap_count = 0
    for genres in recs['listed_in']:
        rec_genres = set(genres.split(', '))
        if input_genres & rec_genres:
            overlap_count += 1

    return overlap_count / top_n

print(genre_overlap_score('Dick Johnson Is Dead'))

1.0


In [20]:
for t in ['Ganglands', 'Midnight Mass', 'Sankofa']:
    print(f"\n--- Recommendations for {t} ---")
    print(get_recommendations(t, top_n=5)[['title', 'listed_in']])


--- Recommendations for Ganglands ---
                       title  \
390   The Eagle of El-Se'eed   
486                The Truth   
576            Fatal Destiny   
6693              Undercover   
6738                   Lupin   

                                              listed_in  
390   Crime TV Shows, International TV Shows, TV Act...  
486   Crime TV Shows, International TV Shows, TV Act...  
576   Crime TV Shows, International TV Shows, TV Act...  
6693  Crime TV Shows, International TV Shows, TV Act...  
6738  Crime TV Shows, International TV Shows, TV Act...  

--- Recommendations for Midnight Mass ---
                           title                           listed_in
6619     Brand New Cherry Flavor  TV Dramas, TV Horror, TV Mysteries
7046   The Haunting of Bly Manor  TV Dramas, TV Horror, TV Mysteries
7076                     Ratched  TV Dramas, TV Horror, TV Mysteries
7910  The Haunting of Hill House  TV Dramas, TV Horror, TV Mysteries
7962               The Originals

In [21]:
def display_recommendations(title, top_n=5):
    recs = get_recommendations(title, top_n=top_n)
    print(f"\n{'='*50}")
    print(f"  Because you watched: {title}")
    print(f"{'='*50}")
    for i, row in recs.iterrows():
        print(f"\n  🎬 {row['title']}")
        print(f"     Genres: {row['listed_in']}")
        print(f"     Director: {row['director']}")
    print(f"\n{'='*50}\n")

display_recommendations('Sankofa')


  Because you watched: Sankofa

  🎬 Kappela
     Genres: Dramas, Independent Movies, International Movies
     Director: Musthafa

  🎬 Mallesham
     Genres: Dramas, Independent Movies, International Movies
     Director: Raj R

  🎬 Genius
     Genres: Dramas, Independent Movies, International Movies
     Director: Suseenthiran

  🎬 Nathicharami
     Genres: Dramas, Independent Movies, International Movies
     Director: Mansore

  🎬 Taramani
     Genres: Dramas, Independent Movies, International Movies
     Director: Ram


